##1.	Exploratory Analysis (30%):
###1)	Distributional analysis, subgroup pattern exploration, and hypothesis generation (minimum 8–10 figures)


In [0]:
import matplotlib.pyplot as mp
import pandas as pd
import seaborn as sb
import numpy as np
import random
from datetime import datetime

In [0]:
df_spark = spark.read.table("workspace.default.spx_with_indicators")

df = df_spark.toPandas()

In [0]:
df.describe()

In [0]:
df.dtypes

## 1) Distribution Analysis

In [0]:
def histogram(x):
    colors = ['blue', 'green', 'red', 'purple', 'orange']
    color = random.choice(colors)
    mp.hist(df[x], color=color, ec='black', bins=15)
    mp.xlabel(x)
    mp.ylabel("Amount")
    mp.title('Distribution of SPX ' + x)
    mp.show()

In [0]:
for column in df:
    histogram(column)

In [0]:
mp.boxplot(df['Volume'])
mp.title("Volume Boxplot")
mp.show()

###2) Subgroup Pattern Exploration

In [0]:
def scatter(x, y, color):
    mp.scatter(df[x], df[y], color=color, marker='o', s=100, alpha=0.7)
    mp.xlabel(x)
    mp.ylabel(y)
    mp.title(f'{y} vs {x}')
    mp.show()

In [0]:
scatter('SMA_200','ret_1d', 'green')

In [0]:
date_column = "Date"
price_column = "Close"

#Specify range
start_date = pd.to_datetime("2021-01-04")
end_half_date = pd.to_datetime("2023-07-04")
end_date = pd.to_datetime("2025-12-31")


df[date_column] = pd.to_datetime(df[date_column], errors='coerce')

mask = (df[date_column] >= start_date) & (df[date_column] <= end_half_date)

mask2 = (df[date_column] > end_half_date) & (df[date_column] <= end_date)

filtered_df = df.loc[mask]

filtered_df2 = df.loc[mask2]

if filtered_df.empty or filtered_df2.empty:
    print("No data found in date range")
else:
    mp.figure(figsize=(10,6))
    mp.plot(filtered_df[date_column], filtered_df[price_column], marker='o', linestyle='-')
    mp.title(f"Prices from {start_date} to {end_half_date}")
    mp.xlabel("Date")
    mp.ylabel("Price($)")
    mp.grid(True)
    mp.tight_layout()
    mp.show()

    mp.figure(figsize=(10,6))
    mp.plot(filtered_df2[date_column], filtered_df2[price_column], marker='o', linestyle='-')
    mp.title(f"Prices from {end_half_date} to {end_date}")
    mp.xlabel("Date")
    mp.ylabel("Price($)")
    mp.grid(True)
    mp.tight_layout()
    mp.show()

In [0]:
# Make sure Date is datetime + sorted
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
df = df.sort_values("Date")

# Regime label (Bull if above SMA_200)
df["Regime"] = (df["Close"] >= df["SMA_200"]).map({True: "Bull", False: "Bear"})

# Boolean masks for shading
is_bull = (df["Regime"] == "Bull").fillna(False)
is_bear = (df["Regime"] == "Bear").fillna(False)

ymin = df["Close"].min()
ymax = df["Close"].max()

mp.figure(figsize=(12, 6))
mp.plot(df["Date"], df["Close"], label="Close")
mp.plot(df["Date"], df["SMA_200"], label="SMA_200")

mp.fill_between(df["Date"], ymin, ymax, where=is_bull, alpha=0.15, label = "Bull")
mp.fill_between(df["Date"], ymin, ymax, where=is_bear, alpha=0.15, label = "Bearish")

mp.title("Bull vs Bear Regimes (Close vs Simple Moving Average 200 Days(SMA_200))")
mp.xlabel("Date")
mp.ylabel("Price ($)")
mp.legend(loc = "lower right")
mp.grid(True)
mp.tight_layout()
mp.show()

In [0]:
df_spark1 = spark.read.table("workspace.default.capstonetrades_closesma200")

df_spark2 = spark.read.table("workspace.default.capstonetrades_sma20_50")

df_spark3 = spark.read.table("workspace.default.capstonetrades_rsi40")

df_spark4 = spark.read.table("workspace.default.capstonetrades_sma20_50_200")


df_CloseSMA200 = df_spark1.toPandas()

df_SMA2050 = df_spark2.toPandas()

df_RSI40 = df_spark3.toPandas()

df_SMA2050200 = df_spark4.toPandas()